# ZCRMSNorm

### How this norm is different

**Three norms you will see:**

**1. LayerNorm (GPT-2, BERT, older code):**
$$y = \gamma \odot \frac{x - \bar{x}}{\sqrt{\mathrm{Var}(x)+\varepsilon}} + \beta$$
Removes mean, rescales, then stretch/shift. Two learned vectors, both usually start near "do nothing" ($\gamma=1,\beta=0$).

**2. RMSNorm (Llama, Gemma, Qwen — general practice):**
$$y = \gamma \odot \frac{x}{\sqrt{\mathrm{mean}(x^2)+\varepsilon}}$$
No mean subtraction, no bias. Only a scale $\gamma$, initialized to 1. At step 0 you get "unit-RMS, then leave it."

**3. Needle ZCRMSNorm (what you just wrote):**
$$y = (1+\gamma)\odot \frac{x}{\sqrt{\mathrm{mean}(x^2)+\varepsilon}},\qquad \gamma \text{ starts at } 0$$
Same formula family as Llama. Two differences only:

| Feature | Llama / Gemma / Qwen RMSNorm | Needle ZCRMSNorm |
| :--- | :--- | :--- |
| **Formula** | $\gamma \cdot x/\mathrm{rms}$ | $(1+\gamma)\cdot x/\mathrm{rms}$ |
| **Init of $\gamma$** | ones | zeros |
| **Step 0 behaviour** | already unit RMS | also unit RMS (1+0) |
| **What training learns** | $\gamma$ away from 1 | $\gamma$ away from 0 |

**Summary:**
So it is not a new normalizer. It is RMSNorm with a zero-init trick so the default is "pass the normalized vector through," and any learned stretch is an increment on top of 1.


In [1]:
from pathlib import Path
import sys

HERE = Path.cwd()
if HERE.name != "question_four":
    candidate = HERE / "open_assessment_one" / "question_four"
    if candidate.is_dir():
        HERE = candidate
sys.path.insert(0, str(HERE))

import needle
print("cwd     :", Path.cwd())
print("needle  :", needle.__file__)

cwd     : d:\1. PhD Presentations\ADL-2026\advanced-deep-learning\open_assessment_one\question_four
needle  : d:\1. PhD Presentations\ADL-2026\advanced-deep-learning\open_assessment_one\question_four\needle.py


In [2]:
import importlib
import torch

import needle
importlib.reload(needle)
from needle import ZCRMSNorm

torch.manual_seed(0)
x = torch.randn(2, 5, 16)
norm = ZCRMSNorm(16)
y = norm(x)

print("in  shape", tuple(x.shape))
print("out shape", tuple(y.shape))
print("scale    ", norm.scale.detach().tolist()[:4], "...")
print("in  rms  ", float(x.pow(2).mean().sqrt()))
print("out rms  ", float(y.pow(2).mean().sqrt()))

assert y.shape == x.shape
assert norm.scale.shape == (16,)
assert torch.allclose(norm.scale, torch.zeros(16))
assert abs(float(y.pow(2).mean().sqrt()) - 1.0) < 0.15
print("ZCRMSNorm test ok")

in  shape (2, 5, 16)
out shape (2, 5, 16)
scale     [0.0, 0.0, 0.0, 0.0] ...
in  rms   0.9871775507926941
out rms   0.9999994039535522
ZCRMSNorm test ok


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_22632\2300614342.py:17: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:821.)
  print("out rms  ", float(y.pow(2).mean().sqrt()))


# RoPE - Rotational Positional Encoding


### Positional Encoding: What people generally do

There are two common options for giving a model a sense of order:

1. **Absolute Positional Embeddings (GPT-2, BERT, older code):** 
   Add a position vector to the embedding: $x_t + p_t$. The model must learn from scratch that $p_3$ means "third token."
2. **Rotary Position Embedding / RoPE (Llama, Gemma, Qwen, Needle):** 
   Do *not* add anything to the input. Instead, rotate the query ($Q$) and key ($K$) of token $t$ by an angle that depends on $t$.

*Note: Needle uses the second option. This is the same idea as current small LMs, not a Needle invention.*

---

#### The Move: Rotate, do not add

Split a head vector into two halves, $a$ and $b$. Treat that as a 2-D plane and rotate it by an angle $\theta_t$:

$$
\begin{pmatrix} a' \\ b' \end{pmatrix}
=
\begin{pmatrix}
\cos\theta_t & -\sin\theta_t \\
\sin\theta_t & \cos\theta_t
\end{pmatrix}
\begin{pmatrix} a \\ b \end{pmatrix}
$$

In code, this is efficiently implemented as:
```python
x_rotated = x * cos_theta + rotate_half(x) * sin_theta

In [3]:
import importlib
import torch

import needle
importlib.reload(needle)
from needle import apply_rope, rope_cos_sin, rotate_half

torch.manual_seed(0)
q = torch.randn(2, 4, 6, 8)
cos, sin = rope_cos_sin(seq_len=6, head_dim=8, device=q.device, dtype=q.dtype)
q_rot = apply_rope(q, cos, sin)

print("q    ", tuple(q.shape))
print("cos  ", tuple(cos.shape))
print("q_rot", tuple(q_rot.shape))

assert q_rot.shape == q.shape
assert cos.shape == (1, 1, 6, 8)
assert torch.allclose(q.norm(dim=-1), q_rot.norm(dim=-1), atol=1e-5)
assert not torch.allclose(q_rot[:, :, 0, :], q_rot[:, :, 5, :], atol=1e-4)
assert rotate_half(q).shape == q.shape
print("RoPE test ok")

q     (2, 4, 6, 8)
cos   (1, 1, 6, 8)
q_rot (2, 4, 6, 8)
RoPE test ok


### Attention Masks: Blocking the wrong keys

Attention is fundamentally "every token looks at every other token." A **mask** is how we say *not those ones*. 

Without a mask, softmax sees the full grid of scores. Two kinds of keys must not win that softmax: **pad**, and **the future**.

---

#### 1. Padding Mask — The batch is a rectangle
A batch cannot hold sentences of different lengths unless we pad the short ones with a special `PAD` token.

```text
"hello world"     →  [hello, world, PAD, PAD]
"good morning ok" →  [good, morning, ok,  PAD]

### Attention Masks: Blocking the wrong keys

Attention is fundamentally "every token looks at every other token." A **mask** is how we say *not those ones*. 

Without a mask, softmax sees the full grid of scores. Two kinds of keys must not win that softmax: **pad**, and **the future**.

---

#### 1. Padding Mask — The batch is a rectangle
A batch cannot hold sentences of different lengths unless we pad the short ones with a special `PAD` token.

```text
"hello world"     →  [hello, world, PAD, PAD]
"good morning ok" →  [good, morning, ok,  PAD]

## 2. Causal mask — the decoder cannot cheat

Training feeds the **whole** target at once (teacher forcing). If token 3 can
see token 4, it copies the answer instead of predicting it. At test time token
4 does not exist yet, so train and test would disagree.

**Rule:** position $t$ may attend to keys $0,\ldots,t$ only.

```text
              key →
            0      1      2      3
query 0    see   BLOCK  BLOCK  BLOCK
query 1    see    see   BLOCK  BLOCK
query 2    see    see    see   BLOCK
query 3    see    see    see    see

## 3. Why merge the masks

Attention scores have shape `(batch, heads, query_len, key_len)`.
The mask that is added to those scores must match that grid.

The two masks we already have do not:

| mask | raw shape | what it marks |
|---|---|---|
| pad | `(batch, key_len)` | this key is padding |
| causal | `(query_len, key_len)` | this key is in the query's future |

`merge_masks` does two things:

1. Broadcast each mask onto `(batch, heads, query_len, key_len)`.
2. Combine them with OR: block the key if **either** mask says block.

```text
score[b, h, t, s] is dropped if
    key s is PAD
    or (decoder) key s is after query t

In [4]:
import importlib
import torch

import needle
importlib.reload(needle)
from needle import make_causal_mask, make_padding_mask, merge_masks

ids = torch.tensor([[3, 4, 0, 0]])
pad = make_padding_mask(ids, pad_id=0)
causal = make_causal_mask(4)
merged = merge_masks(pad, causal, n_heads=2)

print("pad         ", pad.tolist())
print("causal row0 ", causal[0].tolist())
print("merged shape", tuple(merged.shape))

assert pad.tolist() == [[False, False, True, True]]
assert causal[0].tolist() == [False, True, True, True]
assert causal[-1].tolist() == [False, False, False, False]
assert merged.shape == (1, 2, 4, 4)
print("masks test ok")

pad          [[False, False, True, True]]
causal row0  [False, True, True, True]
merged shape (1, 2, 4, 4)
masks test ok


# Scaled Dot-Product Attention: The one formula

For one query vector $q$ and keys/values $k_i, v_i$:

$$
\mathrm{score}_i = \frac{q \cdot k_i}{\sqrt{d}}
\qquad
w_i = \mathrm{softmax}_i(\mathrm{score})
\qquad
o = \sum_i w_i\, v_i
$$

In tensors, that is:

```python
scores  = (Q @ K.transpose(-1, -2)) / math.sqrt(d)  # (batch, heads, Q_len, K_len)
scores  = scores.masked_fill(mask, float('-inf'))   # blocked keys → −∞
weights = softmax(scores, dim=-1)                   # same shape, rows sum to 1
out     = weights @ V                               # (batch, heads, Q_len, head_dim)

In [5]:
import importlib
import torch
import needle
importlib.reload(needle)
from needle import scaled_dot_product_attention

q = torch.randn(1, 1, 3, 4)
k = torch.randn(1, 1, 3, 4)
v = torch.zeros(1, 1, 3, 4)
v[0, 0, 2] = 10.0

mask = torch.zeros(1, 1, 3, 3, dtype=torch.bool)
mask[..., 2] = True
out = scaled_dot_product_attention(q, k, v, mask=mask)

assert out.shape == (1, 1, 3, 4)
assert torch.allclose(out, torch.zeros_like(out), atol=1e-5)
print("attention test ok")

attention test ok


# Grouped Query Attention (GQA): The Inference Bottleneck

To understand why Grouped Query Attention (GQA) exists, we first have to understand the main bottleneck of generating text with a Transformer: **Memory Bandwidth**, not compute.

During autoregressive decoding (generating token by token), the model must load the **KV Cache** (the keys and values of all previously generated tokens) from GPU memory into the compute cores for *every single new token*. 
* The weights ($W_q, W_k, W_v, W_o$) are loaded once per layer.
* The KV cache is loaded **every single step**, and it grows linearly with sequence length and batch size.

If the KV cache is too large, the GPU spends all its time waiting for memory transfers rather than doing math. GQA was invented to shrink this cache without destroying the model's intelligence.

---

#### The Attention Spectrum

To see where GQA fits, let's look at the three main ways to project Queries, Keys, and Values across $H$ attention heads.

**1. Multi-Head Attention (MHA) — The Standard**
Every query head gets its own dedicated key head and value head.
* **Heads:** $H$ Query, $H$ Key, $H$ Value.
* **Pros:** Maximum expressivity. Each head can learn completely independent routing and content.
* **Cons:** Massive KV cache.

```text
MHA (e.g., original Transformer)
Q heads:  q0   q1   q2   q3
          |    |    |    |
K heads:  k0   k1   k2   k3
V heads:  v0   v1   v2   v3
```

**2. Multi-Query Attention (MQA) — The Extreme**
All query heads share a **single** key head and a **single** value head.
* **Heads:** $H$ Query, $1$ Key, $1$ Value.
* **Pros:** Tiny KV cache. Extremely fast inference.
* **Cons:** Severe drop in model quality. Forcing 8 or 16 query heads to share just 1 key/value representation limits what they can learn.

```text
MQA
Q heads:  q0   q1   q2   q3
           \   |   /   /
            \  |  /   /
K head:      [ k_shared ]
V head:      [ v_shared ]
```

**3. Grouped Query Attention (GQA) — The Sweet Spot**
We divide the query heads into $G$ groups. Each group shares **one** key head and **one** value head. 
* **Heads:** $H$ Query, $G$ Key, $G$ Value (where $1 < G < H$).
* **Pros:** Shrinks the KV cache by a factor of $H/G$ while retaining almost all the quality of MHA.

```text
GQA (e.g., Llama 2/3, Needle)
Q heads:  q0   q1   q2   q3
           \   /      \   /
K heads:   k0          k1
V heads:   v0          v1

Group 1: {q0, q1} shares {k0, v0}
Group 2: {q2, q3} shares {k1, v1}
```

---

#### How GQA Works Under the Hood

Mathematically, the attention formula doesn't change at all. The "trick" to GQA happens right before the dot-product attention. 

Because the standard $\mathrm{softmax}(QK^\top / \sqrt{d})V$ equation requires $Q, K,$ and $V$ to have the exact same number of heads, we simply **duplicate** the shared KV heads to match the number of Query heads in memory.

In code, this is usually a function called `_repeat_kv` or `repeat_interleave`.

```python
def repeat_kv(hidden_states, n_rep):
    """
    hidden_states: (batch, n_kv_heads, seq_len, head_dim)
    n_rep: number of times to repeat (e.g., 2 if 4 Q heads and 2 KV heads)
    """
    # Repeat each KV head n_rep times along the head dimension
    return torch.repeat_interleave(hidden_states, repeats=n_rep, dim=1)
```

**The Tensor Shapes:**
Let's say we have `n_heads = 4` and `n_kv_heads = 2`. The repetition factor is `4 / 2 = 2`.

```text
1. Q shape: (batch, 4, seq_len, head_dim)
2. K shape: (batch, 2, seq_len, head_dim)

   --- Apply repeat_kv to K ---

3. K_rep shape: (batch, 4, seq_len, head_dim)  <- Now matches Q!

4. Attention runs normally: softmax(Q @ K_rep.T / sqrt(d)) @ V_rep
```

*Note: This duplication happens in the GPU's fast SRAM/cache. It does not increase the actual memory footprint of the KV cache stored in the main VRAM, which remains at the smaller `n_kv_heads` size.*

---

#### The Trade-offs: Why not always use MHA?

| Feature | MHA (Standard) | GQA (Llama/Needle) | MQA (Extreme) |
| :--- | :---: | :---: | :---: |
| **KV Cache Size** | $1.0\times$ (Largest) | $0.25\times$ to $0.5\times$ | $0.125\times$ (Smallest) |
| **Model Quality** | Excellent | Excellent (near MHA) | Noticeable degradation |
| **Inference Speed** | Slow (Memory bound) | Fast | Very Fast |
| **Parameters ($W_k, W_v$)**| Most | Slightly fewer | Fewest |

**The Verdict:** GQA gives you the memory savings of MQA with the brainpower of MHA. It is currently the industry standard for almost all modern open-weight LLMs (Llama 2/3, Gemma, Qwen, Mistral).

---

#### Summary for our "Needle" Model

In our specific architecture:
1. **Parameters:** We save memory on the $W_k$ and $W_v$ linear projections because they project to a smaller dimension (`n_kv_heads * head_dim` instead of `n_heads * head_dim`).
2. **Decode Speed:** The KV cache is half the size (if using 2 KV heads for 4 Query heads), meaning we can fit larger batch sizes or longer contexts in GPU memory.
3. **Routing:** The Query heads still get to look at the data from 4 distinct perspectives, preserving the model's ability to learn complex, multi-faceted attention patterns.

In [6]:
import importlib
import torch
import needle
importlib.reload(needle)
from needle import GroupedQueryAttention

attn = GroupedQueryAttention(d_model=32, n_heads=4, n_kv_heads=2, use_rope=True)
x = torch.randn(2, 5, 32)
y = attn(x, x, mask=None)
assert y.shape == x.shape

cross = GroupedQueryAttention(d_model=32, n_heads=4, n_kv_heads=2, use_rope=False)
yc = cross(x, torch.randn(2, 7, 32), mask=None)
assert yc.shape == x.shape
print("GQA test ok")

GQA test ok


# Gated Residuals: The Volume Knob for Sub-layers

A residual is the path that says: *keep the old vector, add a change.*

**General practice (Gemma, Qwen, Llama):**
$$x \leftarrow x + F(x)$$
$F$ is attention or an FFN. The new block always writes at full strength.

**Needle multiplies that write by a learned scalar:**
$$x \leftarrow x + \sigma(g)\,F(\mathrm{Norm}(x))$$
That is all `GatedResidual` is.

---

#### What the code is doing

```python
return residual + torch.sigmoid(self.gate) * update

#### Why a plain add is the default

The residual is the model’s memory. If you replace $x$ with $F(x)$, early information dies. Adding keeps *"who I was"* and layers on *"what I just read."* That is why every current LM uses a residual.

**The problem on a small deep net:** 
At step 0, $F$ is random. A full add dumps pure noise into $x$. Depth 12 of that is highly unstable. Big models survive it with careful initialization tricks. A 20–50M parameter encoder–decoder is much less forgiving.

---

#### Why Needle gates it

The gate is a **volume knob per sub-layer**.

```python
return residual + torch.sigmoid(self.gate) * update
```

| $g$ | $\sigma(g)$ | Effect |
| :---: | :---: | :--- |
| $-\infty$ | $0$ | Block is completely skipped |
| $0$ *(init)* | $0.5$ | Half write (safe start) |
| $+\infty$ | $1$ | Same as general practice (full write) |

So a new encoder/decoder layer can come in quietly and turn itself up *only if it helps*. 

Needle’s own study found the gate is almost free in terms of model quality; it is there purely for **stable training**, not for a new kind of attention. This is one of the few Needle-specific pieces. GQA and RoPE are general practice. The gate is not.

In [7]:
import importlib
import torch
import needle
importlib.reload(needle)
from needle import GatedResidual

gate = GatedResidual()
residual = torch.ones(2, 3, 4)
update = torch.ones(2, 3, 4)
mixed = gate(residual, update)

assert torch.allclose(mixed, residual + 0.5 * update)
print("GatedResidual test ok")

GatedResidual test ok


In [8]:
import importlib
import torch
import needle
importlib.reload(needle)
from needle import FeedForward

ffn = FeedForward(d_model=32, d_ff=64, dropout=0.0)
x = torch.randn(2, 5, 32)
y = ffn(x)
assert y.shape == x.shape
print("FeedForward test ok")

FeedForward test ok


## Encoder / Decoder (Needle SAN)

```
src  →  Encoder  →  memory
tgt  →  Decoder(memory)  →  out ```

The Encoder is bidirectional. The Decoder is causal (autoregressive). The Decoder reads the Encoder's output using cross-attention.

Encoder Layer

``` 
x → Norm → GQA self-attn + RoPE → gated residual
→ [FFN] only if use_ffn=True ```

i. Mask applied: Source padding mask only.
ii. No causal mask: Every source word can look at every other source word.

Decoder Layer

``` 
x → Norm → GQA self-attn + RoPE → gated residual
→ [Norm → GQA cross-attn (no RoPE) → gated residual]  if use_cross_attn
→ [FFN]  if use_ffn ```

### Attention Details in the Decoder

| Attention Type | Query ($Q$) | Keys / Values ($K, V$) | Mask Applied |
| :--- | :--- | :--- | :--- |
| **Self-attention** | target | target | causal + target pad |
| **Cross-attention** | target | encoder memory | source pad |

> **Note:** Cross-attention has no RoPE. The source index and target index are on completely different timelines (different clocks), so applying a shared positional rotation would be mathematically incorrect.








In [9]:
import importlib
import torch
import needle
importlib.reload(needle)
from needle import EncoderLayer

x = torch.randn(2, 5, 32)
san = EncoderLayer(d_model=32, n_heads=4, n_kv_heads=2, d_ff=64, use_ffn=False)
classic = EncoderLayer(d_model=32, n_heads=4, n_kv_heads=2, d_ff=64, use_ffn=True)

y_san = san(x, mask=None)
y_classic = classic(x, mask=None)

assert y_san.shape == x.shape
assert y_classic.shape == x.shape
assert hasattr(san, "ffn") is False
assert hasattr(classic, "ffn") is True
assert sum(p.numel() for p in classic.parameters()) > sum(p.numel() for p in san.parameters())
print("EncoderLayer test ok")

EncoderLayer test ok


In [10]:
import importlib
import torch
import needle
importlib.reload(needle)
from needle import DecoderLayer, make_causal_mask, merge_masks

x = torch.randn(2, 4, 32)
memory = torch.randn(2, 6, 32)
self_mask = merge_masks(None, make_causal_mask(4), n_heads=4)

with_cross = DecoderLayer(
    d_model=32, n_heads=4, n_kv_heads=2, use_ffn=False, use_cross_attn=True
)
no_cross = DecoderLayer(
    d_model=32, n_heads=4, n_kv_heads=2, use_ffn=False, use_cross_attn=False
)

y1 = with_cross(x, memory, self_mask, cross_mask=None)
y2 = no_cross(x, memory, self_mask, cross_mask=None)

assert y1.shape == x.shape
assert y2.shape == x.shape
assert hasattr(with_cross, "cross_attn") is True
assert hasattr(no_cross, "cross_attn") is False
print("DecoderLayer test ok")

DecoderLayer test ok


In [11]:
import importlib
import torch
import needle
importlib.reload(needle)
from needle import NeedleBackbone

src = torch.randn(2, 7, 32)
tgt = torch.randn(2, 5, 32)
src_pad = torch.tensor([[False]*5 + [True]*2, [False]*4 + [True]*3])
tgt_pad = torch.tensor([[False]*4 + [True], [False]*5])

for flag in ("all", "one"):
    net = NeedleBackbone(
        n_enc=2, n_dec=3, d_model=32, n_heads=4, n_kv_heads=2,
        use_ffn=False, cross_attn=flag,
    )
    out = net(src, tgt, src_pad, tgt_pad)
    n_cross = sum(layer.use_cross_attn for layer in net.decoder.layers)
    expect = 3 if flag == "all" else 1
    print(flag, tuple(out.shape), "cross_layers", n_cross)
    assert out.shape == (2, 5, 32)
    assert n_cross == expect
print("backbone test ok")

all (2, 5, 32) cross_layers 3
one (2, 5, 32) cross_layers 1
backbone test ok


In [12]:
import importlib
import torch
import model as q4model
importlib.reload(q4model)
from model import TranslationModel, count_parameters

src = torch.randint(3, 50, (2, 6))
tgt = torch.randint(3, 50, (2, 4))
src[:, -1] = 0

for flag in ("all", "one"):
    net = TranslationModel(
        vocab_size=50, d_model=32, n_enc=2, n_dec=2,
        pad_id=0, bos_id=1, eos_id=2, cross_attn=flag,
    )
    logits = net(src, tgt)
    gen = net.generate(src, max_new_tokens=5)
    print(flag, tuple(logits.shape), tuple(gen.shape), count_parameters(net))
    assert logits.shape == (2, 4, 50)
    assert gen.size(0) == 2
print("TranslationModel test ok")

all (2, 4, 50) (2, 6) 20358
one (2, 4, 50) (2, 6) 17221
TranslationModel test ok


In [13]:
import importlib
import data as q4data
importlib.reload(q4data)
from data import PREFIX_EN2NE, PREFIX_NE2EN, SMOKE_PAIRS, expand_pairs, format_pair

src, tgt = format_pair("good morning", "शुभ प्रभात", "en2ne")
print("en2ne", src, "->", tgt)
assert src.startswith(PREFIX_EN2NE) and tgt == "शुभ प्रभात"

src, tgt = format_pair("good morning", "शुभ प्रभात", "ne2en")
print("ne2en", src, "->", tgt)
assert src.startswith(PREFIX_NE2EN) and tgt == "good morning"

rows = expand_pairs(SMOKE_PAIRS)
assert len(rows) == 2 * len(SMOKE_PAIRS)
print("format_pair test ok")

en2ne [en2ne] good morning -> शुभ प्रभात
ne2en [ne2en] शुभ प्रभात -> good morning
format_pair test ok


In [14]:
import importlib
import data as q4data
importlib.reload(q4data)
from data import encode_pair, format_pair, load_tokenizer, special_ids

try:
    tok = load_tokenizer()
except ImportError as exc:
    print("SKIP tokenizer:", exc)
else:
    ids = special_ids(tok)
    print("vocab", ids["vocab_size"], "pad", ids["pad_id"])
    src, tgt = format_pair("good morning", "शुभ प्रभात", "en2ne")
    src_ids, tgt_ids = encode_pair(tok, src, tgt)
    print("src", src, src_ids[:12])
    print("tgt ids", tgt_ids[:12])
    assert tgt_ids[0] == ids["bos_id"] and tgt_ids[-1] == ids["eos_id"]
    print("decode src:", tok.decode(src_ids))
    print("tokenizer test ok")

SKIP tokenizer: Need transformers. On adl-cpu: pip install transformers. On Colab do not pip install requirements.txt.


In [15]:
import importlib
import data as q4data
importlib.reload(q4data)
from data import collate_batch

batch = [
    ([1, 2, 3], [4, 5, 6, 7]),
    ([8, 9], [10, 11, 12]),
]
out = collate_batch(batch, pad_id=0)
print({k: tuple(v.shape) for k, v in out.items()})
print(out["src_ids"])

assert tuple(out["src_ids"].shape) == (2, 3)
assert tuple(out["tgt_in"].shape) == (2, 3)
assert int(out["src_ids"][1, 2]) == 0
print("collate_batch test ok")

{'src_ids': (2, 3), 'tgt_in': (2, 3), 'labels': (2, 3)}
tensor([[1, 2, 3],
        [8, 9, 0]])
collate_batch test ok


In [ ]:
import importlib
import data as q4data
importlib.reload(q4data)
from data import build_dataloaders, load_tokenizer

tok = load_tokenizer()
loaders = build_dataloaders(tok, batch_size=2)
batch = next(iter(loaders["train"]))
print("ids", loaders["ids"])
print({k: tuple(v.shape) for k, v in batch.items()})
assert batch["src_ids"].ndim == 2
assert batch["tgt_in"].shape == batch["labels"].shape
print("dataloader test ok")